# Column-Level Data Dictionary

This notebook walks through every raw dataset (excluding the four wide
gene-expression matrices that have 1000+ gene columns - those are tackled
separately) and explains, in plain language, **what each column means
biologically** and **why it might matter** for selecting a cell line.

For every dataset we build a small `pandas` DataFrame with these columns:

- `dataset` - which raw file this column comes from
- `column` - the name of the field in the raw file
- `dtype` - the pandas data type (inferred from a sample of rows)
- `missingness_pct` - percentage of missing (`NaN`) values, estimated from a
  5,000-row sample of each file
- `description` - a domain-expert explanation of what the field represents
- `suggestion` - preprocessing considerations: what decision we need to make
  about this column (keep/drop/transform/join-key/etc.)

At the end of the notebook all per-dataset dictionaries are concatenated into
one master table and exported to
`CellLineSelector_Data_Dictionary.xlsx` for easy reference while building the
preprocessing pipeline.


In [1]:
import pandas as pd
pd.set_option('display.max_colwidth', 120)
pd.set_option('display.max_rows', 300)

all_dicts = []

def build_dict_df(df, desc, suggestion, dataset_name, columns=None):
    """Build a column/dtype/missingness/description/suggestion table for one dataset."""
    columns = list(columns) if columns is not None else list(df.columns)
    out = pd.DataFrame({'column': columns})
    out['dtype'] = [str(df[c].dtype) for c in columns]
    out['missingness_pct'] = [round(df[c].isna().mean() * 100, 2) for c in columns]
    out['description'] = out['column'].map(desc)
    out['suggestion'] = out['column'].map(suggestion)
    out.insert(0, 'dataset', dataset_name)
    return out


## 1. HPA RNA Cell Line Expression (`1_4_hpa_rna_celline.tsv`)

The Human Protein Atlas (HPA) measured, by RNA sequencing, how much each gene
is "switched on" (expressed) in ~1000 human cancer cell lines. The table is
in **long format**: one row = one gene measured in one cell line. This is the
most direct readout of *which genes a cell line actually uses*, and is the
core signal for matching a cell line to a disease of interest.


In [2]:
df1 = pd.read_csv('../../data/raw/gene_expression/1_4_hpa_rna_celline.tsv', sep='\t', nrows=5000)
desc1 = {
    'Gene': 'Ensembl gene ID (ENSG...) - the stable, version-independent identifier for the gene that was measured.',
    'Gene name': 'HGNC gene symbol (human-readable short name, e.g. TSPAN6) corresponding to the Ensembl ID.',
    'Cell line': 'Name of the cancer cell line in which expression was measured (links to cell-line nomenclature tables).',
    'TPM': 'Transcripts Per Million - raw normalized expression as output by the RNA-seq quantification tool.',
    'pTPM': 'Protein-coding TPM - TPM rescaled so the total sums to one million over protein-coding genes only (expression relative to the protein-coding transcriptome).',
    'nTPM': 'Normalized TPM - HPA batch-corrected value used for fair comparison of a gene\'s expression across different cell lines/experiments.',
}
suggest1 = {
    'Gene': 'Join key to gene-level annotation tables (mutations, fusions); keep for merging, not as a feature itself.',
    'Gene name': 'Redundant with Gene (Ensembl ID); keep as a human-readable label, drop one of the two before modeling.',
    'Cell line': 'Join key to nomenclature tables; standardize naming (case/punctuation/whitespace) before merging.',
    'TPM': 'Raw expression value; log1p-transform before use, and prefer nTPM for cross-cell-line comparisons.',
    'pTPM': 'Highly correlated with TPM; pick a single TPM variant to avoid redundant features.',
    'nTPM': 'Preferred expression feature for cross-cell-line comparison; log1p-transform and pivot to wide format (genes as columns) for modeling.',
}
dict1 = build_dict_df(df1, desc1, suggest1, '1_hpa_rna_celline')
all_dicts.append(dict1)
dict1


,dataset,column,dtype,missingness_pct,description,suggestion
0,1_hpa_rna_celline,Gene,str,0.0,"Ensembl gene ID (ENSG...) - the stable, version-independent identifier for the gene that was measured.","Join key to gene-level annotation tables (mutations, fusions); keep for merging, not as a feature itself."
1,1_hpa_rna_celline,Gene name,str,0.0,"HGNC gene symbol (human-readable short name, e.g. TSPAN6) corresponding to the Ensembl ID.","Redundant with Gene (Ensembl ID); keep as a human-readable label, drop one of the two before modeling."
2,1_hpa_rna_celline,Cell line,str,0.0,Name of the cancer cell line in which expression was measured (links to cell-line nomenclature tables).,Join key to nomenclature tables; standardize naming (case/punctuation/whitespace) before merging.
3,1_hpa_rna_celline,TPM,float64,0.0,Transcripts Per Million - raw normalized expression as output by the RNA-seq quantification tool.,"Raw expression value; log1p-transform before use, and prefer nTPM for cross-cell-line comparisons."
4,1_hpa_rna_celline,pTPM,float64,0.0,Protein-coding TPM - TPM rescaled so the total sums to one million over protein-coding genes only (expression relati...,Highly correlated with TPM; pick a single TPM variant to avoid redundant features.
5,1_hpa_rna_celline,nTPM,float64,0.0,Normalized TPM - HPA batch-corrected value used for fair comparison of a gene's expression across different cell lin...,Preferred expression feature for cross-cell-line comparison; log1p-transform and pivot to wide format (genes as colu...


## 2. Gene Fusion Events (`5_OmicsFusionFilteredSupplementary.csv`)

A **gene fusion** happens when a chromosomal rearrangement joins two
previously separate genes into one. Some fusions create powerful cancer
drivers (e.g. BCR-ABL in leukemia), so detecting them in RNA-seq data tells
us about a cell line's mutational background and which oncogenic pathways
are switched on. Each row is one candidate fusion detected in one sample.


In [3]:
df5 = pd.read_csv('../../data/raw/gene_properties/5_OmicsFusionFilteredSupplementary.csv', nrows=5000)
desc5 = {
    'Unnamed: 0': 'Row index carried over from the source export - no biological meaning.',
    'SequencingID': 'RNA-seq profile ID (PR-xxxx); links this fusion call back to the sequencing profile in DepMap_OmicsProfiles.',
    'ModelID': 'Cell line model identifier (ACH-xxxx); links to DepMap sample info.',
    'IsDefaultEntryForModel': 'Yes/No flag marking the canonical (preferred) profile DepMap uses to represent this model.',
    'ModelConditionID': 'Identifier (MC-xxxx) for the specific experimental condition the sample was processed under.',
    'IsDefaultEntryForMC': 'Yes/No flag marking the canonical profile for that model condition.',
    'CanonicalFusionName': 'Gene1--Gene2 name of the fusion - the two genes that have been joined together.',
    'gene1(ENS ID)': "5' partner gene: its symbol plus its Ensembl gene ID.",
    'gene2(ENS ID)': "3' partner gene: its symbol plus its Ensembl gene ID.",
    'TotalReadsInSample': 'Total number of RNA-seq reads sequenced for this sample - a measure of sequencing depth.',
    'TotalReadsSupportingFusion': 'Number of reads that span or support the fusion junction - evidence strength for the fusion.',
    'FFPM': 'Fusion Fragments Per Million reads - normalized abundance of the fusion, comparable across samples of different depth.',
    'confidence': "The fusion-calling algorithm's confidence tier for this call: high, medium, or low.",
    'split_reads1': 'Number of reads with one segment aligning to gene 1 and the other spanning the breakpoint.',
    'split_reads2': 'Number of reads with one segment aligning to gene 2 and the other spanning the breakpoint.',
    'discordant_mates': 'Paired-end reads whose two mates map to the two different fusion partner genes - additional support for the fusion.',
    'strand1(gene/fusion)': "DNA strand orientation of gene 1 and of the resulting fusion transcript ('+' or '-').",
    'strand2(gene/fusion)': "DNA strand orientation of gene 2 and of the resulting fusion transcript ('+' or '-').",
    'reading_frame': 'Whether the fusion preserves an open reading frame (in-frame) or not (out-of-frame) - in-frame fusions are more likely to produce a functional fusion protein.',
    'breakpoint1': 'Genomic coordinate (chr:position) where gene 1 is broken to form the fusion.',
    'breakpoint2': 'Genomic coordinate (chr:position) where gene 2 is broken to form the fusion.',
    'site1': 'Genomic context of breakpoint 1 (e.g. CDS/splice-site, intergenic, UTR).',
    'site2': 'Genomic context of breakpoint 2 (e.g. CDS/splice-site, intergenic, UTR).',
    'type': 'Structural mechanism that produced the fusion (e.g. deletion/read-through, inversion, translocation).',
    'coverage1': 'Sequencing read depth covering the breakpoint region in gene 1.',
    'coverage2': 'Sequencing read depth covering the breakpoint region in gene 2.',
    'tags': 'Caller-assigned annotation flags; mostly a placeholder value ("." for no tag).',
    'retained_protein_domains': 'Protein domains that remain intact in the predicted fusion protein product.',
    'direction1': "Relative position (upstream/downstream) of gene 1's retained segment within the fusion transcript.",
    'direction2': "Relative position (upstream/downstream) of gene 2's retained segment within the fusion transcript.",
}
suggest5 = {
    'Unnamed: 0': 'Drop - row index with no information.',
    'SequencingID': 'Join key to OmicsProfiles; retain for traceability, not as a model feature.',
    'ModelID': 'Primary join key to cell-line metadata; retain for merging.',
    'IsDefaultEntryForModel': "Filter to 'Yes' to keep one canonical profile per model.",
    'ModelConditionID': 'Secondary join key; usually redundant with ModelID for canonical entries.',
    'IsDefaultEntryForMC': "Filter to 'Yes' to keep one canonical profile per condition.",
    'CanonicalFusionName': 'High-cardinality categorical; better to derive features from gene1/gene2 (e.g. count of fusions per cell line, fusions involving genes of interest).',
    'gene1(ENS ID)': 'Extract gene symbol/Ensembl ID; use to flag/count fusions involving known oncogenes or genes of interest.',
    'gene2(ENS ID)': 'Extract gene symbol/Ensembl ID; use to flag/count fusions involving known oncogenes or genes of interest.',
    'TotalReadsInSample': 'Sequencing-depth covariate; mainly useful as a QC filter, not a direct feature.',
    'TotalReadsSupportingFusion': 'Raw support count; prefer the normalized FFPM value instead.',
    'FFPM': 'Use as the normalized fusion-abundance feature; consider thresholding low-FFPM calls as noise.',
    'confidence': "Ordinal categorical (low/medium/high); consider filtering to medium/high confidence fusions only.",
    'split_reads1': 'QC/support metric; usually summarized into a confidence filter rather than used directly.',
    'split_reads2': 'QC/support metric; usually summarized into a confidence filter rather than used directly.',
    'discordant_mates': 'QC/support metric; usually summarized into a confidence filter rather than used directly.',
    'strand1(gene/fusion)': 'Low-information categorical; likely drop unless studying fusion orientation specifically.',
    'strand2(gene/fusion)': 'Low-information categorical; likely drop unless studying fusion orientation specifically.',
    'reading_frame': 'Important biological flag - in-frame fusions more likely to produce a functional protein; consider as a feature/filter.',
    'breakpoint1': 'High-cardinality coordinate string; not directly usable as a feature, drop or keep only for annotation lookups.',
    'breakpoint2': 'High-cardinality coordinate string; not directly usable as a feature, drop or keep only for annotation lookups.',
    'site1': 'Categorical breakpoint context; useful for filtering to functionally relevant fusions (CDS/splice-site vs intergenic).',
    'site2': 'Categorical breakpoint context; useful for filtering to functionally relevant fusions (CDS/splice-site vs intergenic).',
    'type': 'Low-cardinality categorical mechanism; could be one-hot encoded if fusion mechanism is informative.',
    'coverage1': 'QC metric; usually not a modeling feature.',
    'coverage2': 'QC metric; usually not a modeling feature.',
    'tags': "Mostly placeholder '.'; drop unless populated values carry meaning.",
    'retained_protein_domains': "Mostly placeholder '.'; drop unless populated, otherwise could flag domain-disrupting fusions.",
    'direction1': 'Low-information categorical; likely drop.',
    'direction2': 'Low-information categorical; likely drop.',
}
dict5 = build_dict_df(df5, desc5, suggest5, '5_omics_fusion')
all_dicts.append(dict5)
dict5


,dataset,column,dtype,missingness_pct,description,suggestion
0,5_omics_fusion,Unnamed: 0,int64,0.0,Row index carried over from the source export - no biological meaning.,Drop - row index with no information.
1,5_omics_fusion,SequencingID,str,0.0,RNA-seq profile ID (PR-xxxx); links this fusion call back to the sequencing profile in DepMap_OmicsProfiles.,"Join key to OmicsProfiles; retain for traceability, not as a model feature."
2,5_omics_fusion,ModelID,str,0.0,Cell line model identifier (ACH-xxxx); links to DepMap sample info.,Primary join key to cell-line metadata; retain for merging.
3,5_omics_fusion,IsDefaultEntryForModel,str,0.0,Yes/No flag marking the canonical (preferred) profile DepMap uses to represent this model.,Filter to 'Yes' to keep one canonical profile per model.
4,5_omics_fusion,ModelConditionID,str,0.0,Identifier (MC-xxxx) for the specific experimental condition the sample was processed under.,Secondary join key; usually redundant with ModelID for canonical entries.
5,5_omics_fusion,IsDefaultEntryForMC,str,0.0,Yes/No flag marking the canonical profile for that model condition.,Filter to 'Yes' to keep one canonical profile per condition.
6,5_omics_fusion,CanonicalFusionName,str,0.0,Gene1--Gene2 name of the fusion - the two genes that have been joined together.,"High-cardinality categorical; better to derive features from gene1/gene2 (e.g. count of fusions per cell line, fusio..."
7,5_omics_fusion,gene1(ENS ID),str,0.0,5' partner gene: its symbol plus its Ensembl gene ID.,Extract gene symbol/Ensembl ID; use to flag/count fusions involving known oncogenes or genes of interest.
8,5_omics_fusion,gene2(ENS ID),str,0.0,3' partner gene: its symbol plus its Ensembl gene ID.,Extract gene symbol/Ensembl ID; use to flag/count fusions involving known oncogenes or genes of interest.
9,5_omics_fusion,TotalReadsInSample,int64,0.0,Total number of RNA-seq reads sequenced for this sample - a measure of sequencing depth.,"Sequencing-depth covariate; mainly useful as a QC filter, not a direct feature."


## 3. Somatic Mutations (`6_OmicsSomaticMutationsProfile.csv`)

This is the most detailed dataset: every row is a single DNA mutation found
in a cell line's genome, together with dozens of annotations describing
*where* it is, *what* it changes, and *how dangerous* it is predicted to be
(from multiple independent prediction tools). Mutations in driver genes
(oncogenes/tumor suppressors) are often the **"WHY"** behind choosing a
particular cell line as a disease model - they recreate the genetic lesion
seen in patients.


In [4]:
df6 = pd.read_csv('../../data/raw/gene_properties/6_OmicsSomaticMutationsProfile.csv', nrows=5000, low_memory=False)
desc6 = {
    'Chrom': 'Chromosome on which the variant is located (e.g. chr1).',
    'Pos': 'Genomic position (1-based) of the variant on the chromosome.',
    'Ref': 'Reference allele - the sequence found in the normal human genome.',
    'Alt': 'Alternate (mutant) allele observed in this cell line.',
    'AF': "Allele/variant frequency - the fraction of sequencing reads carrying the mutant allele.",
    'DP': 'Total sequencing read depth at the variant position.',
    'RefCount': 'Number of reads supporting the reference (normal) allele.',
    'AltCount': 'Number of reads supporting the alternate (mutant) allele.',
    'GT': 'Genotype call, e.g. 0/1 = heterozygous, 1/1 = homozygous mutant.',
    'PS': 'Phase set ID - groups variants known to sit on the same chromosome copy (haplotype).',
    'VariantType': 'Class of variant: SNV (single nucleotide), insertion, deletion, substitution, etc.',
    'VariantInfo': "Sequence Ontology term(s) describing the variant's functional effect (e.g. missense_variant).",
    'DNAChange': 'HGVS notation describing the change at the DNA/transcript level (e.g. c.79_80delinsAA).',
    'ProteinChange': 'HGVS notation describing the resulting amino-acid change in the protein (e.g. p.A27K).',
    'HugoSymbol': 'HUGO gene symbol of the gene the mutation falls in.',
    'Exon': 'Which exon is affected, formatted as affected/total (e.g. 1/14).',
    'Intron': 'Which intron is affected, formatted as affected/total.',
    'EnsemblGeneID': 'Ensembl gene identifier of the affected gene.',
    'EnsemblFeatureID': 'Ensembl transcript identifier of the affected transcript.',
    'HgncName': 'Full descriptive name of the gene from HGNC.',
    'HgncFamily': 'HGNC gene family classification (groups genes by shared function/structure).',
    'UniprotID': 'UniProt protein accession/isoform affected by the variant.',
    'DbsnpRsID': 'dbSNP reference SNP ID, if this variant is a previously catalogued polymorphism.',
    'GcContent': 'Local GC nucleotide content around the variant - a sequencing/technical quality indicator.',
    'LofGeneName': 'Name of the gene predicted to lose function because of this variant.',
    'LofGeneId': 'Ensembl ID of the loss-of-function (LoF) gene.',
    'LofNumberOfTranscriptsInGene': 'Total number of annotated transcripts for that gene.',
    'LofPercentOfTranscriptsAffected': 'Fraction of the gene\'s transcripts predicted to be disrupted by this variant.',
    'NMD': 'Nonsense-mediated decay (NMD) prediction details - whether the mutant transcript is likely to be degraded.',
    'MolecularConsequence': 'Sequence Ontology ID and term(s) describing the molecular consequence of the variant.',
    'VepImpact': 'Ensembl VEP severity tier: HIGH, MODERATE, LOW, or MODIFIER.',
    'VepBiotype': 'Biotype of the affected transcript (protein_coding, lncRNA, pseudogene, etc.).',
    'VepHgncID': 'HGNC identifier of the affected gene, as reported by VEP.',
    'VepExistingVariation': 'IDs of known variants (dbSNP/COSMIC) overlapping this position.',
    'VepManeSelect': 'MANE Select transcript - the single, clinically-recommended reference transcript for the gene.',
    'VepENSP': 'Ensembl protein ID (ENSP) of the affected protein.',
    'VepSwissprot': 'UniProt/Swiss-Prot accession of the affected protein.',
    'Sift': 'SIFT prediction of how damaging the amino-acid change is (score + tolerated/deleterious label).',
    'Polyphen': 'PolyPhen-2 prediction of the structural impact on the protein (benign/possibly/probably damaging + score).',
    'GnomadeAF': 'Allele frequency of this variant in the gnomAD exomes population database.',
    'GnomadgAF': 'Allele frequency of this variant in the gnomAD genomes population database.',
    'VepClinSig': 'ClinVar clinical-significance annotation for this variant (e.g. pathogenic, benign).',
    'VepSomatic': 'Flags whether any known overlapping variants are annotated as somatic (cancer-acquired) rather than germline.',
    'VepPliGeneValue': "gnomAD pLI score - probability that this gene is intolerant of loss-of-function mutations.",
    'VepLofTool': 'LoFtool score - a gene-level measure of how intolerant the gene is to loss-of-function variation.',
    'OncogeneHighImpact': 'True/False flag marking this as a high-impact mutation in a known oncogene.',
    'TumorSuppressorHighImpact': 'True/False flag marking this as a high-impact mutation in a known tumor-suppressor gene.',
    'TranscriptLikelyLof': 'List of transcript IDs predicted to be loss-of-function as a result of this variant.',
    'Brca1FuncScore': 'Functional assay score for BRCA1 variants from saturation genome editing experiments.',
    'CivicID': 'Identifier of this variant in the CIViC database of clinically interpreted cancer variants.',
    'CivicDescription': "Curated free-text description from CIViC explaining this variant's clinical relevance.",
    'CivicScore': 'CIViC evidence score - reflects how much/strong clinical evidence exists for this variant.',
    'LikelyLoF': 'Overall True/False call for whether this variant is likely loss-of-function.',
    'HessDriver': 'True/False flag marking this as a likely cancer driver mutation per the Hess et al. signature analysis.',
    'HessSignature': 'Mutational signature(s) (e.g. UV, POLE, CpG) attributed to this variant by Hess et al.',
    'RevelScore': 'REVEL ensemble pathogenicity score for missense variants (combines multiple predictors).',
    'PharmgkbId': 'PharmGKB identifier linking this variant to pharmacogenomic (drug-response) annotations.',
    'DidaID': 'Identifier in the DIDA database of digenic disease associations.',
    'DidaName': 'Disease name associated with this variant via the DIDA database.',
    'GwasDisease': 'Trait or disease this variant has been associated with in genome-wide association studies (GWAS).',
    'GwasPmID': 'PubMed ID of the GWAS publication reporting that association.',
    'GtexGene': 'Gene associated with this variant via GTEx expression quantitative trait loci (eQTL) data.',
    'ProveanPrediction': 'PROVEAN prediction of the variant\'s effect on protein function (Neutral/Damaging).',
    'AMClass': 'AlphaMissense pathogenicity classification (likely_benign / ambiguous / likely_pathogenic).',
    'AMPathogenicity': 'AlphaMissense continuous pathogenicity score, ranging from 0 (benign) to 1 (pathogenic).',
    'Rescue': 'True/False flag indicating the variant was retained ("rescued") by curation rules despite normally being filtered out.',
    'RescueReason': 'Reason code explaining why the variant was rescued/retained (e.g. Oncogene_high_impact).',
    'ProfileID': 'Sequencing profile ID this mutation was called from; links to DepMap_OmicsProfiles.',
    'Hotspot': 'True/False flag marking this position as a known recurrent mutational hotspot in cancer.',
    'EntrezGeneID': 'NCBI Entrez gene ID of the affected gene.',
}
suggest6 = {
    'Chrom': 'Keep for filtering/annotation; not a direct numeric feature.',
    'Pos': 'Genomic coordinate; not a feature on its own, useful for joining/annotation.',
    'Ref': 'Sequence string; usually dropped, retained for variant identity/annotation.',
    'Alt': 'Sequence string; usually dropped, retained for variant identity/annotation.',
    'AF': 'Useful numeric feature - variant allele fraction; can help distinguish clonal vs subclonal mutations.',
    'DP': 'Sequencing-depth QC metric; use to filter low-confidence calls rather than as a feature.',
    'RefCount': 'Redundant with AF/DP; usually drop.',
    'AltCount': 'Redundant with AF/DP; usually drop.',
    'GT': 'Genotype string; could derive a homozygous/heterozygous binary flag.',
    'PS': 'Phasing ID; not useful for modeling, drop.',
    'VariantType': 'Useful categorical for variant-type counts per gene/cell line (SNV/indel/etc.).',
    'VariantInfo': 'Multi-valued consequence string; prefer VepImpact/MolecularConsequence for a cleaner categorical.',
    'DNAChange': 'High-cardinality identifier; drop, retain only for traceability.',
    'ProteinChange': 'High-cardinality identifier; useful for hotspot lookups but not directly as a feature.',
    'HugoSymbol': 'Key join key for gene-level aggregation (e.g. "is gene X mutated in this cell line?").',
    'Exon': 'Low-value positional info; usually drop.',
    'Intron': 'Low-value positional info; usually drop.',
    'EnsemblGeneID': 'Alternative gene join key; redundant with HugoSymbol/EntrezGeneID, pick one.',
    'EnsemblFeatureID': 'Transcript-level identifier; usually drop, too granular.',
    'HgncName': 'Redundant with HugoSymbol; drop.',
    'HgncFamily': 'Could be used to group genes by functional family for pathway-level features.',
    'UniprotID': 'Protein-level identifier; usually drop unless cross-referencing protein databases.',
    'DbsnpRsID': "Mostly used to flag known polymorphisms vs novel somatic variants; consider an 'is_known_snp' binary flag.",
    'GcContent': 'Technical/sequencing covariate; usually drop, not biologically meaningful for cell-line selection.',
    'LofGeneName': "Redundant with HugoSymbol when LikelyLoF is True; could derive an 'is_LoF' flag per gene instead.",
    'LofGeneId': 'Redundant with EnsemblGeneID; drop.',
    'LofNumberOfTranscriptsInGene': 'Gene-level constant; better stored in a gene annotation table than repeated per-mutation.',
    'LofPercentOfTranscriptsAffected': 'Useful severity metric for LoF variants; high missingness expected for non-LoF rows, impute with 0/flag.',
    'NMD': 'Free-text annotation; parse for NMD-escape status if relevant, otherwise drop.',
    'MolecularConsequence': 'Redundant with VariantInfo/VepImpact; pick one consequence representation.',
    'VepImpact': 'Key ordinal feature (HIGH/MODERATE/LOW/MODIFIER) for filtering/weighting mutations by severity.',
    'VepBiotype': "Useful to filter to 'protein_coding' transcripts only.",
    'VepHgncID': 'Redundant gene identifier; drop.',
    'VepExistingVariation': 'Useful to flag variants previously seen in COSMIC/dbSNP (recurrent vs novel).',
    'VepManeSelect': 'Useful to filter to the canonical transcript only, avoiding double-counting per gene.',
    'VepENSP': 'Protein-level identifier; usually drop.',
    'VepSwissprot': 'Protein-level identifier; usually drop unless cross-referencing UniProt.',
    'Sift': 'Parse score and label separately; useful pathogenicity feature for missense variants.',
    'Polyphen': 'Parse score and label separately; useful pathogenicity feature for missense variants.',
    'GnomadeAF': 'Useful to flag rare vs common variants (population frequency); somatic mutations should be rare/absent in gnomAD.',
    'GnomadgAF': 'Same purpose as GnomadeAF (genome vs exome callset); pick one or average to avoid redundancy.',
    'VepClinSig': 'Useful categorical for known pathogenic/benign annotations, but high missingness expected.',
    'VepSomatic': 'Could derive a flag for whether the variant is annotated as somatic in known databases.',
    'VepPliGeneValue': 'Gene-level constraint score; useful for weighting LoF impact by gene essentiality.',
    'VepLofTool': 'Gene-level LoF intolerance score; similar use to VepPliGeneValue, check for redundancy.',
    'OncogeneHighImpact': 'Strong candidate binary feature - directly flags oncogenic driver mutations.',
    'TumorSuppressorHighImpact': 'Strong candidate binary feature - directly flags tumor-suppressor-disrupting mutations.',
    'TranscriptLikelyLof': 'List-valued, redundant with LikelyLoF/LofGeneName; drop.',
    'Brca1FuncScore': 'Extremely high missingness expected (only BRCA1 variants); keep as optional feature with explicit missingness flag.',
    'CivicID': 'Join key to CIViC database; mostly missing, drop unless enriching with CIViC clinical annotations.',
    'CivicDescription': 'Free-text; drop as a model feature, optionally useful for the RAG justification step.',
    'CivicScore': "Sparse clinical-evidence score; could be used as a 'clinically actionable' weight where available.",
    'LikelyLoF': 'Strong candidate binary feature for loss-of-function burden per gene/cell line.',
    'HessDriver': 'Strong candidate binary feature flagging likely cancer driver mutations.',
    'HessSignature': 'Mutational signature labels; could derive per-cell-line signature exposure features.',
    'RevelScore': 'Useful continuous pathogenicity score for missense variants; expect missing values for non-missense rows.',
    'PharmgkbId': 'Mostly missing; drop unless doing pharmacogenomic drug-response analysis.',
    'DidaID': 'Mostly missing; drop unless doing rare-disease association analysis.',
    'DidaName': 'Mostly missing; drop unless doing rare-disease association analysis.',
    'GwasDisease': 'Mostly missing free text; drop as a model feature, optionally useful for justification narrative.',
    'GwasPmID': 'Mostly missing; drop.',
    'GtexGene': 'Mostly missing/empty; drop.',
    'ProveanPrediction': 'Additional pathogenicity predictor; combine with Sift/Polyphen/REVEL into a consensus score or drop if redundant.',
    'AMClass': 'Modern (AlphaMissense) pathogenicity class; strong candidate categorical feature for missense variants.',
    'AMPathogenicity': 'Modern continuous pathogenicity score; strong candidate numeric feature for missense variants.',
    'Rescue': 'Curation flag; useful to understand why a variant was retained, not a core feature.',
    'RescueReason': 'Categorical reason codes; likely redundant with oncogene/tumor-suppressor/hotspot flags captured elsewhere.',
    'ProfileID': 'Join key to OmicsProfiles/sample metadata; retain for merging.',
    'Hotspot': 'Strong candidate binary feature flagging recurrent cancer hotspot mutations.',
    'EntrezGeneID': 'Redundant gene identifier; drop in favor of HugoSymbol/EnsemblGeneID.',
}
dict6 = build_dict_df(df6, desc6, suggest6, '6_omics_mutations')
all_dicts.append(dict6)
dict6


,dataset,column,dtype,missingness_pct,description,suggestion
0,6_omics_mutations,Chrom,str,0.00,Chromosome on which the variant is located (e.g. chr1).,Keep for filtering/annotation; not a direct numeric feature.
1,6_omics_mutations,Pos,int64,0.00,Genomic position (1-based) of the variant on the chromosome.,"Genomic coordinate; not a feature on its own, useful for joining/annotation."
2,6_omics_mutations,Ref,str,0.00,Reference allele - the sequence found in the normal human genome.,"Sequence string; usually dropped, retained for variant identity/annotation."
3,6_omics_mutations,Alt,str,0.00,Alternate (mutant) allele observed in this cell line.,"Sequence string; usually dropped, retained for variant identity/annotation."
4,6_omics_mutations,AF,float64,0.00,Allele/variant frequency - the fraction of sequencing reads carrying the mutant allele.,Useful numeric feature - variant allele fraction; can help distinguish clonal vs subclonal mutations.
5,6_omics_mutations,DP,int64,0.00,Total sequencing read depth at the variant position.,Sequencing-depth QC metric; use to filter low-confidence calls rather than as a feature.
6,6_omics_mutations,RefCount,int64,0.00,Number of reads supporting the reference (normal) allele.,Redundant with AF/DP; usually drop.
7,6_omics_mutations,AltCount,int64,0.00,Number of reads supporting the alternate (mutant) allele.,Redundant with AF/DP; usually drop.
8,6_omics_mutations,GT,str,0.00,"Genotype call, e.g. 0/1 = heterozygous, 1/1 = homozygous mutant.",Genotype string; could derive a homozygous/heterozygous binary flag.
9,6_omics_mutations,PS,float64,91.64,Phase set ID - groups variants known to sit on the same chromosome copy (haplotype).,"Phasing ID; not useful for modeling, drop."


## 4. Cellosaurus Cell Line Catalogue (`7_cellosaurus.csv`)

[Cellosaurus](https://www.cellosaurus.org/) is the master encyclopedia of
cell lines used in biomedical research. It is the "phone book" that lets us
resolve the many different names a cell line might be called across datasets
(GEO, DepMap, HPA, ...) down to one stable accession (`CVCL_xxxx`), and it
carries rich metadata about the donor, disease, and authenticity of each
line.


In [5]:
df7 = pd.read_csv('../../data/raw/nomenclature/7_cellosaurus.csv', nrows=5000, low_memory=False)
desc7 = {
    'Identifier (cell line name)': 'The primary, preferred name used to refer to this cell line.',
    'Accession (CVCL_xxxx)': 'The unique, stable Cellosaurus accession number for this cell line - the universal join key.',
    'Secondary accession number(s)': 'Older/merged Cellosaurus accessions that now redirect to this entry.',
    'Synonyms': 'Alternative names and spellings this cell line is also known by.',
    'Cross-references': "Identifiers for this cell line in other databases (e.g. Wikidata, ATCC, ECACC).",
    'References identifiers': 'Publications (PubMed) or patents that describe or establish this cell line.',
    'Web pages': 'External web links with further information about the cell line.',
    'Comments': 'Free-text curator notes: provenance, transformation status, antibody targets, group classification, etc.',
    'STR profile data': 'Short Tandem Repeat (STR) DNA fingerprint - used to verify the cell line\'s identity and detect cross-contamination/misidentification.',
    'Diseases': 'Disease(s) the cell line is associated with, coded against the NCI Thesaurus ontology.',
    'Species of origin': 'Species the cell line was derived from (e.g. Homo sapiens, Mus musculus), with NCBI taxonomy ID.',
    'Hierarchy': 'Parent cell line(s) this line was derived from, if it is a sub-clone or derivative.',
    'Originate from same individual': 'Other cell lines known to come from the same patient/donor.',
    'Sex of cell': 'Sex of the donor the cell line was derived from.',
    'Age of donor at sampling': "Donor's age (or developmental stage, e.g. Embryo) at the time the sample was taken.",
    'Category': 'High-level category of the cell line (e.g. Cancer cell line, Hybridoma, Transformed cell line).',
    'Date (entry history)': 'Creation date, last-update date, and version number of this Cellosaurus record.',
}
suggest7 = {
    'Identifier (cell line name)': 'Primary join key for cell-line name resolution; standardize formatting (case/whitespace) before merging.',
    'Accession (CVCL_xxxx)': 'Canonical cell-line ID; use as the master join key across all datasets (DepMap RRID, HPA Cellosaurus ID, GEO Cellosaurus_ID).',
    'Secondary accession number(s)': 'Useful for resolving merged/renamed records during ID matching; not a feature.',
    'Synonyms': 'Useful for fuzzy-matching cell line names across datasets; not a feature.',
    'Cross-references': 'Free-text external IDs; drop as a feature, optional for traceability.',
    'References identifiers': 'Free-text citations; drop.',
    'Web pages': 'Free-text links; drop.',
    'Comments': "Free-text; mine for keywords (e.g. 'misidentified', 'contaminated') as a data-quality flag, otherwise drop.",
    'STR profile data': "Useful for verifying cell-line identity/authenticity; could derive an 'STR available' binary flag.",
    'Diseases': "Useful disease annotation; parse NCIt codes/names and cross-check against DepMap 'primary_disease'.",
    'Species of origin': 'Filter to Homo sapiens only before modeling (drop non-human lines).',
    'Hierarchy': 'Useful for excluding sub-clones/derivatives that duplicate a parent line; parse for relationship features.',
    'Originate from same individual': 'Use to group related cell lines by donor; helps prevent data leakage across train/test splits.',
    'Sex of cell': 'Categorical demographic feature; encode as categorical, handle "unspecified" as its own category.',
    'Age of donor at sampling': 'Free-text/mixed format (e.g. "1-2M", "Embryo"); parse to numeric age where possible, bucket otherwise.',
    'Category': "Filter to 'Cancer cell line' (or other relevant categories) before modeling.",
    'Date (entry history)': 'Metadata only; drop as a feature.',
}
dict7 = build_dict_df(df7, desc7, suggest7, '7_cellosaurus')
all_dicts.append(dict7)
dict7


,dataset,column,dtype,missingness_pct,description,suggestion
0,7_cellosaurus,Identifier (cell line name),str,0.00,"The primary, preferred name used to refer to this cell line.",Primary join key for cell-line name resolution; standardize formatting (case/whitespace) before merging.
1,7_cellosaurus,Accession (CVCL_xxxx),str,0.00,"The unique, stable Cellosaurus accession number for this cell line - the universal join key.","Canonical cell-line ID; use as the master join key across all datasets (DepMap RRID, HPA Cellosaurus ID, GEO Cellosa..."
2,7_cellosaurus,Secondary accession number(s),str,99.68,Older/merged Cellosaurus accessions that now redirect to this entry.,Useful for resolving merged/renamed records during ID matching; not a feature.
3,7_cellosaurus,Synonyms,str,72.40,Alternative names and spellings this cell line is also known by.,Useful for fuzzy-matching cell line names across datasets; not a feature.
4,7_cellosaurus,Cross-references,str,3.16,"Identifiers for this cell line in other databases (e.g. Wikidata, ATCC, ECACC).","Free-text external IDs; drop as a feature, optional for traceability."
5,7_cellosaurus,References identifiers,str,13.20,Publications (PubMed) or patents that describe or establish this cell line.,Free-text citations; drop.
6,7_cellosaurus,Web pages,str,93.48,External web links with further information about the cell line.,Free-text links; drop.
7,7_cellosaurus,Comments,str,0.92,"Free-text curator notes: provenance, transformation status, antibody targets, group classification, etc.","Free-text; mine for keywords (e.g. 'misidentified', 'contaminated') as a data-quality flag, otherwise drop."
8,7_cellosaurus,STR profile data,str,95.36,Short Tandem Repeat (STR) DNA fingerprint - used to verify the cell line's identity and detect cross-contamination/m...,Useful for verifying cell-line identity/authenticity; could derive an 'STR available' binary flag.
9,7_cellosaurus,Diseases,str,80.16,"Disease(s) the cell line is associated with, coded against the NCI Thesaurus ontology.",Useful disease annotation; parse NCIt codes/names and cross-check against DepMap 'primary_disease'.


## 5. DepMap Omics Profiles (`8_DepMap_OmicsProfiles.csv`)

DepMap (the Cancer Dependency Map) sequences each cell line in multiple ways
- whole genome (WGS), whole exome (WES), and RNA. This small table is the
**index/lookup** that says, for every sequencing profile, which cell line it
came from and which technology was used. Other DepMap tables (mutations,
fusions, signatures) reference the `ProfileID` defined here.


In [6]:
df8 = pd.read_csv('../../data/raw/nomenclature/8_DepMap_OmicsProfiles.csv', nrows=5000)
desc8 = {
    'ProfileID': 'Unique sequencing profile identifier (PR-xxxx) - the join key used by mutation, fusion, and signature tables.',
    'ModelCondition': 'Identifier (MC-xxxx) for the specific experimental condition the cell line was grown/sequenced under.',
    'ModelID': 'Cell line model identifier (ACH-xxxx) - links to DepMap sample info.',
    'Datatype': 'Sequencing assay performed: wgs (whole genome), wes (whole exome), or rna (RNA-seq).',
    'WESKit': 'Exome capture kit used for whole-exome sequencing (e.g. AGILENT, ICE); empty for non-WES profiles.',
}
suggest8 = {
    'ProfileID': 'Join key to mutation/fusion/signature tables; retain for merging.',
    'ModelCondition': 'Secondary join key; usually redundant with ModelID for default profiles.',
    'ModelID': 'Primary join key to cell-line metadata; retain for merging.',
    'Datatype': 'Use to select the correct profile type (rna/wes/wgs) when joining to other omics tables.',
    'WESKit': 'Technical covariate relevant only to WES rows; expect high missingness for rna/wgs rows, usually drop.',
}
dict8 = build_dict_df(df8, desc8, suggest8, '8_depmap_omics_profiles')
all_dicts.append(dict8)
dict8


,dataset,column,dtype,missingness_pct,description,suggestion
0,8_depmap_omics_profiles,ProfileID,str,0.00,"Unique sequencing profile identifier (PR-xxxx) - the join key used by mutation, fusion, and signature tables.",Join key to mutation/fusion/signature tables; retain for merging.
1,8_depmap_omics_profiles,ModelCondition,str,0.00,Identifier (MC-xxxx) for the specific experimental condition the cell line was grown/sequenced under.,Secondary join key; usually redundant with ModelID for default profiles.
2,8_depmap_omics_profiles,ModelID,str,0.00,Cell line model identifier (ACH-xxxx) - links to DepMap sample info.,Primary join key to cell-line metadata; retain for merging.
3,8_depmap_omics_profiles,Datatype,str,0.00,"Sequencing assay performed: wgs (whole genome), wes (whole exome), or rna (RNA-seq).",Use to select the correct profile type (rna/wes/wgs) when joining to other omics tables.
4,8_depmap_omics_profiles,WESKit,str,51.44,"Exome capture kit used for whole-exome sequencing (e.g. AGILENT, ICE); empty for non-WES profiles.","Technical covariate relevant only to WES rows; expect high missingness for rna/wgs rows, usually drop."


## 6. DepMap Sample Info (`9_DepMap_sample_info.csv`)

This is the **central metadata table** for every cell line in DepMap: who it
is, what cancer it represents, where it came from, and how it relates to
other lines. `DepMap_ID` is the primary key that almost every other DepMap
table (mutations, fusions, signatures, omics profiles) links back to. This
table is the natural place to look for the **"WHY"** - the disease context
that justifies selecting a given cell line.


In [7]:
df9 = pd.read_csv('../../data/raw/nomenclature/9_DepMap_sample_info.csv', nrows=5000, low_memory=False)
desc9 = {
    'DepMap_ID': 'Unique DepMap model identifier (ACH-xxxx) - the primary key linking all DepMap datasets.',
    'cell_line_name': 'Common/published name of the cell line.',
    'stripped_cell_line_name': 'Cell line name with spaces, dashes and punctuation removed - used for fuzzy matching across datasets.',
    'CCLE_Name': 'CCLE naming convention combining the cell line name with its tissue of origin (e.g. SLR21_KIDNEY).',
    'alias': 'Alternative name(s)/synonyms for the cell line.',
    'COSMICID': 'Identifier for this cell line in the COSMIC cancer mutation database.',
    'sex': 'Sex of the donor the cell line was derived from.',
    'source': 'Repository or lab that supplied the cell line (e.g. ATCC, DSMZ, Academic lab).',
    'RRID': 'Research Resource Identifier - the Cellosaurus accession (CVCL_xxxx) for this cell line.',
    'WTSI_Master_Cell_ID': 'Internal cell line identifier used by the Wellcome Sanger Institute.',
    'sample_collection_site': 'Anatomical site the tumor sample was collected from.',
    'primary_or_metastasis': 'Whether the sample was taken from the primary tumor or from a metastatic site.',
    'primary_disease': 'High-level cancer type/diagnosis (e.g. Kidney Cancer, Leukemia).',
    'Subtype': 'More specific disease subtype/histology (e.g. Renal Cell Carcinoma).',
    'age': 'Age of the donor at the time of sample collection.',
    'Sanger_Model_ID': 'Identifier for this cell line in the Sanger Cell Model Passports database.',
    'depmap_public_comments': 'Curator notes/caveats about the cell line, e.g. known misidentification or relationships to other lines.',
    'lineage': 'Broad tissue-of-origin lineage classification used for stratified analyses (e.g. kidney, blood, lung).',
    'lineage_subtype': 'More specific cancer subtype within the lineage (e.g. renal_cell_carcinoma, NSCLC).',
    'lineage_sub_subtype': 'Further refinement of the subtype classification (e.g. NSCLC_adenocarcinoma).',
    'lineage_molecular_subtype': 'Molecular subtype classification based on transcriptional/genomic profile (e.g. basal_B).',
    'default_growth_pattern': 'How the cell line grows in culture: adherent, suspension, or mixed.',
    'model_manipulation': 'Genetic or experimental manipulation applied to create this model, if any (e.g. immortalized, drug resistance selection).',
    'model_manipulation_details': 'Specific details of the manipulation (e.g. "STAG2 KO", "Drug resistance: Dabrafenib and Trametinib").',
    'patient_id': 'Identifier linking multiple cell lines derived from the same patient.',
    'parent_depmap_id': 'DepMap_ID of the parent line this model was derived from, for engineered derivatives.',
    'Cellosaurus_NCIt_disease': 'Disease term for this cell line, mapped via Cellosaurus to the NCI Thesaurus ontology.',
    'Cellosaurus_NCIt_id': 'NCI Thesaurus concept ID corresponding to the disease term.',
    'Cellosaurus_issues': 'Known data-quality or identity issues for this cell line as flagged in Cellosaurus.',
}
suggest9 = {
    'DepMap_ID': 'Primary key for the whole project; use to join all DepMap-derived tables.',
    'cell_line_name': 'Human-readable label; keep for display/justification, not as a model feature.',
    'stripped_cell_line_name': "Best join key for matching against other datasets' cell-line name fields.",
    'CCLE_Name': 'Alternative join key (matches miRNA/metabolomics CCLE_ID); useful for merging those tables.',
    'alias': 'Useful for fuzzy name matching; not a feature.',
    'COSMICID': 'External ID; useful only if cross-referencing the COSMIC database directly.',
    'sex': "Categorical demographic feature; encode and treat missing as its own 'Unknown' category.",
    'source': 'Categorical provenance; could indicate batch effects between repositories, consider as covariate.',
    'RRID': 'Cellosaurus accession; primary join key to the Cellosaurus catalogue.',
    'WTSI_Master_Cell_ID': 'Internal Sanger ID; mostly missing, drop.',
    'sample_collection_site': "Useful tissue-of-origin feature; check consistency with 'lineage'.",
    'primary_or_metastasis': "Useful categorical feature for tumor-stage context; high missingness expected, add an 'Unknown' category.",
    'primary_disease': "Core disease label - central to the 'WHY' justification; primary candidate for filtering/stratification.",
    'Subtype': 'Finer disease label; combine with primary_disease for more specific stratification.',
    'age': 'Mixed string/numeric; parse to numeric age, bucket or impute missing values.',
    'Sanger_Model_ID': 'Join key to Cell Model Passports if additional Sanger data is used; otherwise drop.',
    'depmap_public_comments': "Free-text caveats; mine for 'misidentified'/'derivative of' flags as data-quality filters.",
    'lineage': 'Primary tissue-lineage categorical feature for stratification.',
    'lineage_subtype': 'Finer-grained lineage categorical feature; many levels, consider grouping rare categories.',
    'lineage_sub_subtype': 'Very fine-grained; high cardinality and missingness, use cautiously or drop.',
    'lineage_molecular_subtype': 'Molecular subtype categorical; sparse, could be a useful feature where present.',
    'default_growth_pattern': 'Useful practical feature - indicates ease of culture (adherent vs suspension), relevant to lab feasibility.',
    'model_manipulation': 'Flags genetically engineered lines; consider excluding manipulated derivatives if seeking "natural" disease models.',
    'model_manipulation_details': 'Free-text detail; parse for specific knockouts/edits if relevant, otherwise drop.',
    'patient_id': 'Use to group related cell lines and avoid leakage across train/test splits.',
    'parent_depmap_id': 'Use to identify/exclude derivative lines that duplicate a parent model.',
    'Cellosaurus_NCIt_disease': 'Cross-check against primary_disease for consistency; could fill gaps in disease labeling.',
    'Cellosaurus_NCIt_id': 'Ontology ID; useful for standardized disease matching/hierarchies.',
    'Cellosaurus_issues': 'Free-text data-quality flags; use to filter out problematic cell lines.',
}
dict9 = build_dict_df(df9, desc9, suggest9, '9_depmap_sample_info')
all_dicts.append(dict9)
dict9


,dataset,column,dtype,missingness_pct,description,suggestion
0,9_depmap_sample_info,DepMap_ID,str,0.00,Unique DepMap model identifier (ACH-xxxx) - the primary key linking all DepMap datasets.,Primary key for the whole project; use to join all DepMap-derived tables.
1,9_depmap_sample_info,cell_line_name,str,5.00,Common/published name of the cell line.,"Human-readable label; keep for display/justification, not as a model feature."
2,9_depmap_sample_info,stripped_cell_line_name,str,0.05,"Cell line name with spaces, dashes and punctuation removed - used for fuzzy matching across datasets.",Best join key for matching against other datasets' cell-line name fields.
3,9_depmap_sample_info,CCLE_Name,str,0.22,CCLE naming convention combining the cell line name with its tissue of origin (e.g. SLR21_KIDNEY).,Alternative join key (matches miRNA/metabolomics CCLE_ID); useful for merging those tables.
4,9_depmap_sample_info,alias,str,93.86,Alternative name(s)/synonyms for the cell line.,Useful for fuzzy name matching; not a feature.
5,9_depmap_sample_info,COSMICID,float64,46.68,Identifier for this cell line in the COSMIC cancer mutation database.,External ID; useful only if cross-referencing the COSMIC database directly.
6,9_depmap_sample_info,sex,str,5.54,Sex of the donor the cell line was derived from.,Categorical demographic feature; encode and treat missing as its own 'Unknown' category.
7,9_depmap_sample_info,source,str,2.55,"Repository or lab that supplied the cell line (e.g. ATCC, DSMZ, Academic lab).","Categorical provenance; could indicate batch effects between repositories, consider as covariate."
8,9_depmap_sample_info,RRID,str,1.20,Research Resource Identifier - the Cellosaurus accession (CVCL_xxxx) for this cell line.,Cellosaurus accession; primary join key to the Cellosaurus catalogue.
9,9_depmap_sample_info,WTSI_Master_Cell_ID,float64,46.74,Internal cell line identifier used by the Wellcome Sanger Institute.,"Internal Sanger ID; mostly missing, drop."


## 7. GEO Sample Info (`10_GEOInfo.txt`)

[GEO](https://www.ncbi.nlm.nih.gov/geo/) (Gene Expression Omnibus) is a
public archive of gene-expression experiments, mostly from older microarray
technology. This table is the metadata index for the GEO samples used in
`3_GEOexpression.txt` (the wide expression matrix we are deferring): it
records which GEO sample (`GSM` ID) corresponds to which cell line, and maps
that cell line to a stable Cellosaurus accession.


In [8]:
df10 = pd.read_csv('../../data/raw/nomenclature/10_GEOInfo.txt', sep='\t', nrows=5000, low_memory=False)
desc10 = {
    'Geo_accession': 'GEO sample accession (GSM number) - the unique identifier for one microarray sample.',
    'CEL_file_names': 'Name of the raw microarray (.CEL) data file associated with this sample.',
    'title': 'Free-text title given to the sample by the submitting lab.',
    'status': 'Public release status of the record in GEO, including the date it was made public.',
    'submission_date': 'Date the sample record was submitted to GEO.',
    'last_update_date': 'Date the GEO record was last updated.',
    'type': 'Type of molecule profiled (e.g. RNA).',
    'channel_count': 'Number of detection channels on the array; 1 means a single-channel (one-colour) array.',
    'source_name_ch1': 'Description of the biological source material loaded into channel 1 of the array.',
    'organism_ch1': 'Organism the sample was derived from.',
    'characteristics_ch1': 'Free-text key:value pairs describing sample characteristics (e.g. "gender: male", "cell line: ...").',
    'platform_id': 'GEO platform accession identifying the specific microarray chip used (e.g. GPL570).',
    'contact_country': 'Country of the laboratory that submitted the data.',
    'contact_institute': 'Institution that submitted the data.',
    'GSE_ID': 'GEO Series accession - groups all samples that belong to the same study.',
    'GSE_filename': 'Filename of the series matrix file containing this study\'s full expression data.',
    'cell_line': 'Cell line name exactly as reported by the submitting lab (may be inconsistent/non-standard).',
    'disease': 'Disease/diagnosis associated with the cell line as reported by the submitting lab.',
    'origin': 'Tissue of origin as reported by the submitting lab.',
    'Cellosaurus_ID': 'Cellosaurus accession (CVCL_xxxx) that this cell line has been matched to.',
    'Cellline': 'Standardized cell line name resolved via the Cellosaurus match.',
    'Matching_Type': 'How the reported cell line name was matched to a Cellosaurus entry (e.g. by GSM record, exact name, or synonym).',
    'cell_line_Trimmed': 'Lowercase, punctuation-stripped version of the cell line name, used as a join key for matching across datasets.',
}
suggest10 = {
    'Geo_accession': 'Join key to GEO expression matrix (file 3); retain for merging.',
    'CEL_file_names': 'Redundant with Geo_accession in most rows; drop.',
    'title': 'Free-text; drop as a feature.',
    'status': 'Metadata only; drop.',
    'submission_date': 'Metadata only; drop unless studying batch/era effects.',
    'last_update_date': 'Metadata only; drop.',
    'type': "Constant ('RNA'); drop, no information.",
    'channel_count': 'Constant (1); drop, no information.',
    'source_name_ch1': 'Often redundant with cell_line; use to validate the cell_line field, otherwise drop.',
    'organism_ch1': 'Filter to Homo sapiens; otherwise constant, drop.',
    'characteristics_ch1': 'Free-text key:value pairs; parse for useful attributes (e.g. gender) if needed, otherwise drop.',
    'platform_id': 'Technical covariate (microarray platform); important for batch-effect correction if combining across platforms.',
    'contact_country': 'Metadata only; drop unless studying geographic/lab batch effects.',
    'contact_institute': 'Metadata only; drop unless studying lab batch effects.',
    'GSE_ID': 'Study/batch identifier; useful for batch-effect grouping or train/test splitting by study.',
    'GSE_filename': 'Technical pointer to source file; drop.',
    'cell_line': "Raw, non-standardized cell-line name; prefer 'Cellline' (matched) for joins.",
    'disease': 'Raw disease label as reported; cross-check against DepMap primary_disease, may be inconsistent.',
    'origin': 'Raw tissue label as reported; cross-check against DepMap lineage.',
    'Cellosaurus_ID': 'Best join key to Cellosaurus/DepMap via RRID.',
    'Cellline': "Standardized cell-line name; preferred join key over raw 'cell_line'.",
    'Matching_Type': 'QC field indicating match confidence; could filter out low-confidence matches.',
    'cell_line_Trimmed': 'Normalized join key; useful for fuzzy matching across datasets.',
}
dict10 = build_dict_df(df10, desc10, suggest10, '10_geo_info')
all_dicts.append(dict10)
dict10


,dataset,column,dtype,missingness_pct,description,suggestion
0,10_geo_info,Geo_accession,str,0.00,GEO sample accession (GSM number) - the unique identifier for one microarray sample.,Join key to GEO expression matrix (file 3); retain for merging.
1,10_geo_info,CEL_file_names,str,0.00,Name of the raw microarray (.CEL) data file associated with this sample.,Redundant with Geo_accession in most rows; drop.
2,10_geo_info,title,str,28.71,Free-text title given to the sample by the submitting lab.,Free-text; drop as a feature.
3,10_geo_info,status,str,28.71,"Public release status of the record in GEO, including the date it was made public.",Metadata only; drop.
4,10_geo_info,submission_date,str,28.71,Date the sample record was submitted to GEO.,Metadata only; drop unless studying batch/era effects.
5,10_geo_info,last_update_date,str,28.71,Date the GEO record was last updated.,Metadata only; drop.
6,10_geo_info,type,str,28.71,Type of molecule profiled (e.g. RNA).,"Constant ('RNA'); drop, no information."
7,10_geo_info,channel_count,float64,28.71,Number of detection channels on the array; 1 means a single-channel (one-colour) array.,"Constant (1); drop, no information."
8,10_geo_info,source_name_ch1,str,28.71,Description of the biological source material loaded into channel 1 of the array.,"Often redundant with cell_line; use to validate the cell_line field, otherwise drop."
9,10_geo_info,organism_ch1,str,28.71,Organism the sample was derived from.,"Filter to Homo sapiens; otherwise constant, drop."


## 8. HPA Cell Line Description (`11_hpa_rna_celline_description.tsv`)

A short, curated metadata table for each cell line in the HPA RNA expression
dataset (Section 1). It supplies the disease context and donor information
needed to interpret the gene expression values - i.e. it answers "what
disease does this cell line model, and where did it come from?"


In [9]:
df11 = pd.read_csv('../../data/raw/nomenclature/11_hpa_rna_celline_description.tsv', sep='\t', nrows=5000)
desc11 = {
    'Cell line': 'Name of the cell line, matching the "Cell line" column in the HPA RNA expression dataset.',
    'Disease': 'Cancer type the cell line represents (e.g. Bone cancer, Prostate cancer).',
    'Disease subtype': 'More specific histological subtype of the disease (e.g. Osteosarcoma, Adenocarcinoma).',
    'Cellosaurus ID': 'Cellosaurus accession (CVCL_xxxx) for cross-referencing with other datasets.',
    'Patient': "Donor demographic information where known (age and/or sex), e.g. 'Male, 72'.",
    'Primary/Metastasis': 'Whether the source tumor sample was from the primary site or a metastasis.',
    'Sample collection site': 'Anatomical site the tumor sample was taken from.',
}
suggest11 = {
    'Cell line': 'Join key to HPA expression dataset; standardize formatting before merging with other tables.',
    'Disease': 'Disease label; cross-check against DepMap primary_disease for consistency.',
    'Disease subtype': 'Finer disease label; combine with Disease for stratification.',
    'Cellosaurus ID': 'Preferred join key to Cellosaurus/DepMap via RRID.',
    'Patient': 'Inconsistent free-text (age/sex mixed); parse into separate age/sex fields, high missingness expected.',
    'Primary/Metastasis': 'Useful categorical; inconsistent capitalization, normalize before encoding. High missingness expected.',
    'Sample collection site': 'Useful tissue-of-origin feature; cross-check against DepMap sample_collection_site.',
}
dict11 = build_dict_df(df11, desc11, suggest11, '11_hpa_description')
all_dicts.append(dict11)
dict11


,dataset,column,dtype,missingness_pct,description,suggestion
0,11_hpa_description,Cell line,str,0.00,"Name of the cell line, matching the ""Cell line"" column in the HPA RNA expression dataset.",Join key to HPA expression dataset; standardize formatting before merging with other tables.
1,11_hpa_description,Disease,str,0.00,"Cancer type the cell line represents (e.g. Bone cancer, Prostate cancer).",Disease label; cross-check against DepMap primary_disease for consistency.
2,11_hpa_description,Disease subtype,str,13.18,"More specific histological subtype of the disease (e.g. Osteosarcoma, Adenocarcinoma).",Finer disease label; combine with Disease for stratification.
3,11_hpa_description,Cellosaurus ID,str,0.66,Cellosaurus accession (CVCL_xxxx) for cross-referencing with other datasets.,Preferred join key to Cellosaurus/DepMap via RRID.
4,11_hpa_description,Patient,str,8.71,"Donor demographic information where known (age and/or sex), e.g. 'Male, 72'.","Inconsistent free-text (age/sex mixed); parse into separate age/sex fields, high missingness expected."
5,11_hpa_description,Primary/Metastasis,str,28.11,Whether the source tumor sample was from the primary site or a metastasis.,"Useful categorical; inconsistent capitalization, normalize before encoding. High missingness expected."
6,11_hpa_description,Sample collection site,str,5.64,Anatomical site the tumor sample was taken from.,Useful tissue-of-origin feature; cross-check against DepMap sample_collection_site.


## 9. Omics Global Signatures (`14_OmicsGlobalSignatures.csv`)

These are **genome-wide summary statistics** computed from each cell line's
whole-genome/exome sequencing - single numbers that describe how "chaotic" a
cancer genome is overall (instability, ploidy, mismatch-repair status). They
are useful as quick, high-level features for comparing cell lines without
needing to look at individual mutations.


In [10]:
df14 = pd.read_csv('../../data/raw/non_gene_expression/14_OmicsGlobalSignatures.csv', nrows=5000)
desc14 = {
    'Unnamed: 0': 'Row index carried over from the source export - no biological meaning.',
    'SequencingID': 'WGS/WES profile ID (PR-xxxx); links to DepMap_OmicsProfiles.',
    'ModelID': 'Cell line model identifier (ACH-xxxx); links to DepMap sample info.',
    'ModelConditionID': 'Identifier (MC-xxxx) for the specific experimental condition the sample was processed under.',
    'IsDefaultEntryForModel': 'Yes/No flag marking the canonical profile DepMap uses to represent this model.',
    'IsDefaultEntryForMC': 'Yes/No flag marking the canonical profile for that model condition.',
    'MSIScore': 'Microsatellite instability score; high values indicate a defective DNA mismatch-repair system, a known driver of hypermutation.',
    'LoHFraction': 'Fraction of the genome showing Loss of Heterozygosity (where one parental copy of a chromosome region has been lost).',
    'WGD': 'Whole-Genome Doubling flag (0/1) - whether the cell line\'s genome appears to have doubled in its evolutionary history.',
    'CIN': 'Chromosomal Instability score - summarizes how much copy-number variation/heterogeneity exists across the genome.',
    'Ploidy': 'Estimated average number of chromosome copies per cell (normal human cells are diploid, ploidy ~2).',
    'Aneuploidy': 'Score/count of chromosome arms with abnormal (non-diploid) copy number.',
}
suggest14 = {
    'Unnamed: 0': 'Drop - row index with no information.',
    'SequencingID': 'Join key to OmicsProfiles; retain for merging.',
    'ModelID': 'Primary join key to cell-line metadata.',
    'ModelConditionID': 'Secondary join key; usually redundant with ModelID for default entries.',
    'IsDefaultEntryForModel': "Filter to 'Yes' to keep one canonical entry per model.",
    'IsDefaultEntryForMC': "Filter to 'Yes' to keep one canonical entry per condition.",
    'MSIScore': 'Useful continuous feature; check distribution, may be bimodal (MSI-high vs MSI-stable).',
    'LoHFraction': 'Useful continuous genomic-instability feature; moderate missingness expected, consider imputation.',
    'WGD': 'Binary genomic feature; moderate missingness expected.',
    'CIN': 'Useful continuous genomic-instability feature; correlated with LoHFraction/Aneuploidy, check multicollinearity.',
    'Ploidy': 'Useful continuous feature; correlated with WGD, check multicollinearity.',
    'Aneuploidy': 'Useful continuous feature; correlated with CIN/Ploidy, check multicollinearity.',
}
dict14 = build_dict_df(df14, desc14, suggest14, '14_omics_global_signatures')
all_dicts.append(dict14)
dict14


,dataset,column,dtype,missingness_pct,description,suggestion
0,14_omics_global_signatures,Unnamed: 0,int64,0.00,Row index carried over from the source export - no biological meaning.,Drop - row index with no information.
1,14_omics_global_signatures,SequencingID,str,0.00,WGS/WES profile ID (PR-xxxx); links to DepMap_OmicsProfiles.,Join key to OmicsProfiles; retain for merging.
2,14_omics_global_signatures,ModelID,str,0.00,Cell line model identifier (ACH-xxxx); links to DepMap sample info.,Primary join key to cell-line metadata.
3,14_omics_global_signatures,ModelConditionID,str,0.00,Identifier (MC-xxxx) for the specific experimental condition the sample was processed under.,Secondary join key; usually redundant with ModelID for default entries.
4,14_omics_global_signatures,IsDefaultEntryForModel,str,0.00,Yes/No flag marking the canonical profile DepMap uses to represent this model.,Filter to 'Yes' to keep one canonical entry per model.
5,14_omics_global_signatures,IsDefaultEntryForMC,str,0.00,Yes/No flag marking the canonical profile for that model condition.,Filter to 'Yes' to keep one canonical entry per condition.
6,14_omics_global_signatures,MSIScore,float64,0.00,"Microsatellite instability score; high values indicate a defective DNA mismatch-repair system, a known driver of hyp...","Useful continuous feature; check distribution, may be bimodal (MSI-high vs MSI-stable)."
7,14_omics_global_signatures,LoHFraction,float64,14.53,Fraction of the genome showing Loss of Heterozygosity (where one parental copy of a chromosome region has been lost).,"Useful continuous genomic-instability feature; moderate missingness expected, consider imputation."
8,14_omics_global_signatures,WGD,float64,14.53,Whole-Genome Doubling flag (0/1) - whether the cell line's genome appears to have doubled in its evolutionary history.,Binary genomic feature; moderate missingness expected.
9,14_omics_global_signatures,CIN,float64,14.53,Chromosomal Instability score - summarizes how much copy-number variation/heterogeneity exists across the genome.,"Useful continuous genomic-instability feature; correlated with LoHFraction/Aneuploidy, check multicollinearity."


## 10. CCLE Metabolomics (`12_CCLE_metabolomics_20190502.csv`)

Metabolomics measures the relative abundance of small-molecule metabolites
(sugars, amino acids, lipids, etc.) in each cell line - essentially a
snapshot of its biochemical activity. This complements gene-expression data:
two cell lines with similar gene expression can still have very different
metabolic behaviour, which can matter for drug response.

Most of the 225 metabolite columns are named either with a common metabolite
name (e.g. `glucose`, `lactate`) or with **lipid shorthand** of the form
`Cxx:y CLASS`, where `xx` is the total number of carbons in the lipid's fatty
acid chains, `y` is the number of double bonds (unsaturations), and `CLASS`
is the lipid class (e.g. `PC` = phosphatidylcholine, `TAG` = triacylglycerol,
`Cer` = ceramide).


In [11]:
df12 = pd.read_csv('../../data/raw/non_gene_expression/12_CCLE_metabolomics_20190502.csv', nrows=5000)

LIPID_CLASSES = {
    'PC': 'phosphatidylcholine', 'PE': 'phosphatidylethanolamine', 'PI': 'phosphatidylinositol',
    'PS': 'phosphatidylserine', 'PG': 'phosphatidylglycerol', 'PA': 'phosphatidic acid',
    'TAG': 'triacylglycerol (fat storage)', 'DAG': 'diacylglycerol', 'MAG': 'monoacylglycerol',
    'Cer': 'ceramide (sphingolipid)', 'SM': 'sphingomyelin', 'LPC': 'lyso-phosphatidylcholine',
    'LPE': 'lyso-phosphatidylethanolamine', 'CE': 'cholesteryl ester', 'FA': 'free fatty acid',
    'CL': 'cardiolipin (mitochondrial membrane lipid)', 'HCER': 'hexosylceramide', 'LCER': 'lactosylceramide',
}

def describe_metabolite(name):
    """Return a domain description for a metabolite/lipid column name."""
    parts = name.split()
    if len(parts) == 2 and ':' in parts[0]:
        chain, cls = parts
        cls_full = LIPID_CLASSES.get(cls, cls)
        return f"Lipid species ({cls_full}, shorthand '{cls}') with total acyl-chain composition {chain} (carbons:double bonds). Relative abundance measured by mass spectrometry."
    return f"Relative abundance of the metabolite '{name}', measured by mass spectrometry (LC-MS)."

desc12 = {
    'CCLE_ID': 'Legacy CCLE sample identifier combining cell-line name and tissue (e.g. NAME_TISSUE).',
    'DepMap_ID': 'Stable DepMap model identifier (ACH-xxxx); the preferred join key to other DepMap tables.',
}
suggest12 = {
    'CCLE_ID': 'Redundant with DepMap_ID for joining; can drop once merged, or keep for human-readable labels.',
    'DepMap_ID': 'Primary join key to other DepMap tables; retain.',
}
for col in df12.columns:
    if col in desc12:
        continue
    desc12[col] = describe_metabolite(col)
    suggest12[col] = 'Continuous abundance feature; check missingness and scale/normalize (e.g. log-transform) before modeling. Highly correlated lipid species may need dimensionality reduction.'

dict12 = build_dict_df(df12, desc12, suggest12, '12_ccle_metabolomics')
all_dicts.append(dict12)
dict12.head(10)


,dataset,column,dtype,missingness_pct,description,suggestion
0,12_ccle_metabolomics,CCLE_ID,str,0.00,Legacy CCLE sample identifier combining cell-line name and tissue (e.g. NAME_TISSUE).,"Redundant with DepMap_ID for joining; can drop once merged, or keep for human-readable labels."
1,12_ccle_metabolomics,DepMap_ID,str,0.11,Stable DepMap model identifier (ACH-xxxx); the preferred join key to other DepMap tables.,Primary join key to other DepMap tables; retain.
2,12_ccle_metabolomics,2-aminoadipate,float64,0.00,"Relative abundance of the metabolite '2-aminoadipate', measured by mass spectrometry (LC-MS).",Continuous abundance feature; check missingness and scale/normalize (e.g. log-transform) before modeling. Highly cor...
3,12_ccle_metabolomics,3-phosphoglycerate,float64,0.00,"Relative abundance of the metabolite '3-phosphoglycerate', measured by mass spectrometry (LC-MS).",Continuous abundance feature; check missingness and scale/normalize (e.g. log-transform) before modeling. Highly cor...
4,12_ccle_metabolomics,alpha-glycerophosphate,float64,0.00,"Relative abundance of the metabolite 'alpha-glycerophosphate', measured by mass spectrometry (LC-MS).",Continuous abundance feature; check missingness and scale/normalize (e.g. log-transform) before modeling. Highly cor...
5,12_ccle_metabolomics,4-pyridoxate,float64,0.00,"Relative abundance of the metabolite '4-pyridoxate', measured by mass spectrometry (LC-MS).",Continuous abundance feature; check missingness and scale/normalize (e.g. log-transform) before modeling. Highly cor...
6,12_ccle_metabolomics,aconitate,float64,0.00,"Relative abundance of the metabolite 'aconitate', measured by mass spectrometry (LC-MS).",Continuous abundance feature; check missingness and scale/normalize (e.g. log-transform) before modeling. Highly cor...
7,12_ccle_metabolomics,adenine,float64,0.00,"Relative abundance of the metabolite 'adenine', measured by mass spectrometry (LC-MS).",Continuous abundance feature; check missingness and scale/normalize (e.g. log-transform) before modeling. Highly cor...
8,12_ccle_metabolomics,adipate,float64,0.00,"Relative abundance of the metabolite 'adipate', measured by mass spectrometry (LC-MS).",Continuous abundance feature; check missingness and scale/normalize (e.g. log-transform) before modeling. Highly cor...
9,12_ccle_metabolomics,alpha-ketoglutarate,float64,0.00,"Relative abundance of the metabolite 'alpha-ketoglutarate', measured by mass spectrometry (LC-MS).",Continuous abundance feature; check missingness and scale/normalize (e.g. log-transform) before modeling. Highly cor...


## 11. CCLE miRNA Expression (`13_CCLE_miRNA_20181103.gct`)

MicroRNAs (miRNAs) are small RNA molecules that regulate gene expression
after transcription - they can silence or fine-tune the genes measured in
the mRNA expression datasets (Sections 1-2). This file is in the **GCT**
format used by the Broad Institute: it has two header lines, then a table
where each row is a miRNA probe and each of the 954 remaining columns is one
cell line's expression value for that miRNA. Because nearly all of those 954
columns share an identical structure (one cell-line expression column), they
are summarized here as a single representative row rather than listed
individually.


In [12]:
with open('../../data/raw/non_gene_expression/13_CCLE_miRNA_20181103.gct', encoding='utf-8') as f:
    f.readline()  # version line
    dims_line = f.readline()  # n_rows \t n_samples
    header = f.readline().strip().split('\t')

n_mirna_rows, n_samples = (int(x) for x in dims_line.strip().split('\t'))
sample_cols = header[2:]

df13 = pd.read_csv('../../data/raw/non_gene_expression/13_CCLE_miRNA_20181103.gct', sep='\t', skiprows=2, nrows=5000)

desc13 = {
    'Name': 'miRBase accession ID for the mature microRNA probe (e.g. MIMAT0000xxx).',
    'Description': 'Human-readable miRNA name (e.g. hsa-miR-21-5p) indicating the species (hsa = human) and miRNA family/arm.',
    '<cell line columns>': f'One column per cell line ({n_samples} total, e.g. {sample_cols[0]}, {sample_cols[1]}, ...), named CELLLINE_TISSUE. Each value is the normalized expression level of that miRNA in that cell line.',
}
suggest13 = {
    'Name': 'Stable identifier for the miRNA feature; keep as row index after transposing to cell-line-as-row format.',
    'Description': 'Human-readable miRNA name; useful for biological annotation/lookups, can drop for modeling.',
    '<cell line columns>': 'Transpose so each cell line is a row and each miRNA is a feature column before joining with other omics tables on cell-line name. Check missingness per cell line and normalize/scale before modeling.',
}

sample_missingness = round(df13[sample_cols].isna().mean().mean() * 100, 2)
dict13 = pd.DataFrame({
    'dataset': '13_ccle_mirna',
    'column': ['Name', 'Description', '<cell line columns>'],
    'dtype': [str(df13['Name'].dtype), str(df13['Description'].dtype), str(df13[sample_cols[0]].dtype)],
    'missingness_pct': [
        round(df13['Name'].isna().mean() * 100, 2),
        round(df13['Description'].isna().mean() * 100, 2),
        sample_missingness,
    ],
    'description': [desc13['Name'], desc13['Description'], desc13['<cell line columns>']],
    'suggestion': [suggest13['Name'], suggest13['Description'], suggest13['<cell line columns>']],
})
all_dicts.append(dict13)
dict13


,dataset,column,dtype,missingness_pct,description,suggestion
0,13_ccle_mirna,Name,str,0.0,miRBase accession ID for the mature microRNA probe (e.g. MIMAT0000xxx).,Stable identifier for the miRNA feature; keep as row index after transposing to cell-line-as-row format.
1,13_ccle_mirna,Description,str,0.0,Human-readable miRNA name (e.g. hsa-miR-21-5p) indicating the species (hsa = human) and miRNA family/arm.,"Human-readable miRNA name; useful for biological annotation/lookups, can drop for modeling."
2,13_ccle_mirna,<cell line columns>,float64,0.0,"One column per cell line (954 total, e.g. DMS53_LUNG, SW1116_LARGE_INTESTINE, ...), named CELLLINE_TISSUE. Each valu...",Transpose so each cell line is a row and each miRNA is a feature column before joining with other omics tables on ce...


## 12. Combined Data Dictionary

Concatenating all per-dataset tables above into a single master data
dictionary, with one row per raw column across all 11 datasets. This gives a
single reference sheet for: what each column means, how complete it is, and
what to consider when building the preprocessing pipeline.


In [13]:
final_df = pd.concat(all_dicts, ignore_index=True)
final_df.to_excel('CellLineSelector_Data_Dictionary.xlsx', index=False, sheet_name='data_dictionary')
print(f'Total columns documented: {len(final_df)}')
final_df


Total columns documented: 429


,dataset,column,dtype,missingness_pct,description,suggestion
0,1_hpa_rna_celline,Gene,str,0.0,"Ensembl gene ID (ENSG...) - the stable, version-independent identifier for the gene that was measured.","Join key to gene-level annotation tables (mutations, fusions); keep for merging, not as a feature itself."
1,1_hpa_rna_celline,Gene name,str,0.0,"HGNC gene symbol (human-readable short name, e.g. TSPAN6) corresponding to the Ensembl ID.","Redundant with Gene (Ensembl ID); keep as a human-readable label, drop one of the two before modeling."
2,1_hpa_rna_celline,Cell line,str,0.0,Name of the cancer cell line in which expression was measured (links to cell-line nomenclature tables).,Join key to nomenclature tables; standardize naming (case/punctuation/whitespace) before merging.
3,1_hpa_rna_celline,TPM,float64,0.0,Transcripts Per Million - raw normalized expression as output by the RNA-seq quantification tool.,"Raw expression value; log1p-transform before use, and prefer nTPM for cross-cell-line comparisons."
4,1_hpa_rna_celline,pTPM,float64,0.0,Protein-coding TPM - TPM rescaled so the total sums to one million over protein-coding genes only (expression relati...,Highly correlated with TPM; pick a single TPM variant to avoid redundant features.
...,...,...,...,...,...,...
424,12_ccle_metabolomics,C58:7 TAG,float64,0.0,"Lipid species (triacylglycerol (fat storage), shorthand 'TAG') with total acyl-chain composition C58:7 (carbons:doub...",Continuous abundance feature; check missingness and scale/normalize (e.g. log-transform) before modeling. Highly cor...
425,12_ccle_metabolomics,C58:6 TAG,float64,0.0,"Lipid species (triacylglycerol (fat storage), shorthand 'TAG') with total acyl-chain composition C58:6 (carbons:doub...",Continuous abundance feature; check missingness and scale/normalize (e.g. log-transform) before modeling. Highly cor...
426,13_ccle_mirna,Name,str,0.0,miRBase accession ID for the mature microRNA probe (e.g. MIMAT0000xxx).,Stable identifier for the miRNA feature; keep as row index after transposing to cell-line-as-row format.
427,13_ccle_mirna,Description,str,0.0,Human-readable miRNA name (e.g. hsa-miR-21-5p) indicating the species (hsa = human) and miRNA family/arm.,"Human-readable miRNA name; useful for biological annotation/lookups, can drop for modeling."
